# A basic introduction to PyTorch and Neural Networks

Tratto dalla summer school su "PINN" (Physics Informed Neural Networks), KTH 2023: https://pinns.se/

Credit: Prof. Khemraj Shukla

In [ ]:
# Installa PyTorch per CPU:
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cpu

In [ ]:
# OPZIONALE: in alternativa, non installare pyTorch per CPU e controlla di avere una GPU NVIDIA:
!nvcc --version

In [ ]:
# Oppure:
!nvidia-smi

In [ ]:
# Se hai una GPU NVIDIA, puoi installare la versione di pyTorch corrispondente a CUDA
# https://pytorch.org/get-started/locally/

# Rimovi '#' e cambia "cu126" con la tua versione di CUDA
# !pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126

## Breve introduzione a PyTorch

NumPy è di _fatto necessario_ per utilizzare PyTorch:

In [ ]:
import numpy as np
import torch

Impostiamo il seed, per garantire la riproducibilità dei risultati:

In [ ]:
torch.manual_seed(42)

### Operazioni su tensori

Il tipo di dato PyTorch per i vettori è `tensor` ("tensore" in senso generalizzato: vettore, matrice, tensore, ...).

In [ ]:
torch.tensor([[4, 5, 6], [1, 7, 8]])

I metodi sono molto simili a quelli di PyTorch:

In [ ]:
# Matrice di zeri (3,3)
torch.zeros((3,3))

In [ ]:
# Matrice identita' (2,2)
torch.eye(2)

In [ ]:
# Matrice randomica (3,3)
torch.rand(3,3)

In [ ]:
# Scalari
s = torch.tensor(1.)
print(f"Scalar x: {s}")
d = s.dim()
print(f"Dimension of scalar is: {d}")

# Vettori
v = torch.tensor([1., 2., 3.])
print(f"Vector v: {v}")
d = v.dim()
print(f"Dimension of vector is: {d}")

# Matrici
m = torch.tensor([[1., 2., 3.],[4., 5., 6.]])
print(f"Matrix m: {m}")
d = m.dim()
print(f"Dimension of matrix is: {d}")

# Tensori
t = torch.tensor([[[1., 2., 3.],[4., 5., 6.], [1., 2., 3.],[4., 5., 6.]]])
print(f"Tensor t: {t}")
d = t.dim()
print(f"Dimension of tensor is: {d}")

Somma su tensori, `dim=1` indica che la somma è per colonna:

In [ ]:
x = torch.rand(3,2)
print(f"x: {x}")

xsum = torch.sum(x, dim=1)
print(f"Sum using method1: {xsum}")

xsum = x.sum(dim=1)
print(f"Sum using method2: {xsum}")

"Reshaping", ovvero come passare da un vettore 1D a una matrice 2D etc...

In [ ]:
# Usando "reshape" (copia in un nuovo array)
x = torch.tensor([1,2,3,4,5,6,7,8,9,10,11,12])
# Matrice (3,4)
y3 = x.reshape(3,4)
print(y3)
# Matrice con un numero di righe non specificato e 4 colonne
y4 = x.reshape(-1,4)
print(y4)

# Usando "view" (piu' veloce, puo' essere usato solo se i dati sono "vicini" in memoria)
x = torch.tensor([1,2,3,4,5,6,7,8,9,10, 11, 12])
y1 = x.view(3,4)
print(y1)
y2 = x.view(-1,4)
print(y2)

Norme tensoriali (ricordate il termine di penalizzazione nella Ridge/LASSO regression?):

In [ ]:
x = torch.rand(3)
x.norm(p=1)
x.norm(p=2)
print(f"L1 Norm of x is:{x.norm(p=1)}")
print(f"L2 Norm of x is:{x.norm(p=2)}")

In [ ]:
# Manualmente...
n1 = torch.sum(torch.abs(x))
print(f"L1 norm: is: {n1}")
n2 = torch.sqrt(torch.sum(x**2))
print(f"L2 norm: is: {n2}")

# ...o chiamando i metodi degli oggetti
n1 = x.abs().sum()
print(f"L1 norm: is: {n1}")
n2 = (x**2).sum().sqrt() 
print(f"L2 norm: is: {n2}")

### Trasferire dati a CPU/GPU

In [ ]:
# Controlla se hai una GPU NVIDIA (CUDA),
# e hai installato PyTorch con supporto GPU
torch.cuda.is_available()

In [ ]:
x = torch.tensor([[4,5,8], [3,8,9]])

# Se hai una GPU, de-commenta la seconda riga:
dev_cpu = torch.device("cpu")
# dev_gpu = torch.device("cuda:0")

# Se hai una GPU, puoi provare a mandare il tensore alla GPU
x.to(dev_cpu)
# x.to(dev_gpu)

In [ ]:
# Salva il dispositivo (CPU oppure GPU)
device = torch.device("cpu" if not torch.cuda.is_available() else "cuda")
print("Device:",device)

# Inviamo (o meglio: programmiamo di inviare) il tensore 'x' al dispositivo
x.to(device)

In [ ]:
# Da/a NumPy, a/da PyTorch

x = np.random.random((4,4))
print(type(x))

Y = torch.from_numpy(x)
print(type(Y))

X = Y.numpy()
print(type(X))

In [ ]:
# Timing su CPU

A = torch.rand(100, 400, 400)
A.size()
%timeit -n 3 torch.bmm(A, A)

In [ ]:
# Timing su GPU (se disponibile)

if torch.cuda.is_available() :
    B = A.cuda()
    %timeit -n 3 torch.bmm(B, B)
else :
    print("No CUDA GPU available")

### Piccola digressione

Per gestire l'ordine delle somme su tensori 3D, PyTorch offre il metodo `einsum()` che definisce la somma usando la [notazione di Einstein](https://en.wikipedia.org/wiki/Einstein_notation) (gli indici ripetuti sono sommati).

In [ ]:
A=torch.randint(3, 10, (3, 4))
B=torch.randint(3, 10, (4, 3))

C = torch.matmul(A, B)
print(C)

# Somma secondo la notazione di Einstein (l'indice 'j' scompare, ma l'ordine e' diverso!)
C = torch.einsum("ij,jk->ik", A, B)
print(C)
C = torch.einsum("ij,jk->ki", A, B)
print(C)

In [ ]:
A=torch.randint(3, 10, (3, 4, 3))
B=torch.randint(3, 10, (3, 3, 4))

D = torch.matmul(A, B)
print(D)

# Equivalente a torch.matmul(A, B)
D = torch.einsum("bij,bjk->bik", A, B)
print(D)

# Qui invece l'ordine e' diverso!
D = torch.einsum("bij,bjk->ikb", A, B)
print(D)

## La nostra prima rete neurale

In [ ]:
!pip3 install torchsummary
from torchsummary import summary

In [ ]:
import torch.nn.functional as F
import torch.utils.data as Data
from torch.autograd import Variable
from torch.optim import Adam, LBFGS, SGD
from torch.utils.data import Dataset, DataLoader

In [ ]:
import imageio
import matplotlib.pyplot as plt
from IPython.display import Image, display
%matplotlib inline

**Obiettivo**: creare ed allenare una semplice rete neurale (1 layer) per risolvere un problema di regressione.

Creiamo il dataset:

In [ ]:
# Seed (riproducibilita')
torch.manual_seed(1234)

# Numero di punti nel dataset
N_p = 100

# Unsqueeze aggiunge una dimensione, i.e. (N_p,)->(N_p,1)
# La rete neurale vuole in vettore 2D in input
x = torch.unsqueeze(torch.linspace(-1, 1, N_p), dim=1)
print(x.shape)

# Parabola + rumore (uniforme)
y = torch.square(x)
y = y + 0.1*torch.rand(y.size())

plt.figure(figsize=(8,4))
x_plot, y_plot = x.numpy(), y.numpy()
plt.scatter(x_plot, y_plot, color="red")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Data for Regression Analysis")
plt.show()

### Architettura

Creiamo la nostra "shallow" neural network:
- No. hidden layers = 1
- Architettura: sequenziale, lineare, feed-forward
- Activation functions: [ReLU](https://en.wikipedia.org/wiki/Rectified_linear_unit)

<img src="https://docs.pytorch.org/docs/2.11/_images/ReLU.png" width="450"/>

$$\text{ReLU}(x)=\text{max}(0,x)$$

In [ ]:
# no. input = no. output = 1, perche' e' un problema di regressione in 1D
in_dim = 1
out_dim = 1

# Numero di neuroni nell'hidden layer
n_neur = 40

# Oggetto PyTorch che definisce la rete neurale
Net = torch.nn.Sequential(
      torch.nn.Linear(in_dim, n_neur),
      torch.nn.ReLU(),
      torch.nn.Linear(n_neur, out_dim))

# Qualche informazione
summary(Net, (100, 1))

###  Funzione obiettivo e ottimizzatore

Come funzione obiettivo, usiamo una "semplice" Mean Square Loss (MSE), di fatto la stessa usata per nell'esempio di regressione polinomiale:

$$ \text{MSE} = \frac{1}{N}\sum_{i=0}^N[y_i-f(x_i)]^2 $$

In [ ]:
loss_function = torch.nn.MSELoss()

Ottimizzatore = algoritmo che implementa Stochastic Gradient Descent (SGD).

Formulazione generale:

$$ \boldsymbol{w}_{k+1} = \boldsymbol{w}_k -\eta\nabla_{\boldsymbol{w}}\mathcal{L}(\boldsymbol{w}) \; , \quad \mathcal{L}(\boldsymbol{w})=\text{MSE}(\boldsymbol{w}) $$

Dove $\eta$ è chiamato _learning rate_, e calibra la velocità di discesa (e.g. per "scappare" da minimi locali).

Useremo una versione "migliorata" di SGD, chiamata [ADAptive Moment estimation (ADAM)](https://arxiv.org/abs/1412.6980).
- _Momentum optimization_, aggiunge un termine di attrito in modo da smorzare le oscillazioni del SGD, permettere di uscire rapidamente dai plateau della funzione obiettivo;
- _Adaptive_, poiché tiene traccia della media dei gradienti precedenti per ri-definire il temine di attrito.

In [ ]:
# lr = learning rate
optimizer = torch.optim.Adam(Net.parameters(), lr=0.01)

### Training

In [ ]:
# Lista di immagini (che poi trasformeremo in .gif)
image_list = []

# Numero di iterazioni di training
Niter = 150 + 1

fig, ax = plt.subplots(figsize=(15,10))

for it in range(Niter):
    
    # Valutiamo la rete neurale
    y_pred = Net(x)
    # Calcoliamo l'errore (loss)
    loss = loss_function(y_pred, y)
    # Puliamo i parametri dell'algoritmo di SGD
    optimizer.zero_grad()
    # Back-propagation (i.e. calcola il gradiente)
    loss.backward()
    # Step di ottimizzazione (ADAM)
    # e ottimizzazione dei parametri
    optimizer.step()
    
    if it % 50 == 0:
        print(f"Doing Iterations: {it} and Loss: {loss}")
    
    ### Salviamo il plot per ogni iterazione ###
    plt.cla()
    ax.set_xlabel('x', fontsize=32)
    ax.set_ylabel('y Predictied', fontsize=32)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.scatter(x_plot, y_plot, color = "red")
    ax.plot(x_plot, y_pred.data.numpy(), 'k-', lw=4)
    ax.text(-0.5, 1.3, 'It = %d,' %it, fontdict={'size': 22, 'color':  'black'})
    ax.text(-0.1, 1.3, 'Loss = %.6f' % loss.data.numpy(),
            fontdict={'size': 22, 'color':  'black'})
    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    image = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    image = image.reshape((h, w, 4))
    image = image[:, :, :3].copy()
    image_list.append(image)
    ### ------------------------------------ ###

In [ ]:
# Salviamo la lista di immagini come .gif
print("### Salvo .gif... ###")
imageio.mimsave('./Parabolic_regression_1.gif', image_list, fps=5)

### Secondo esempio - regressione su funzione periodica

In [ ]:
x = torch.unsqueeze(torch.linspace(-6, 6, 100), dim=1) 

# Sinusoide + rumore
y = torch.sin(x)
y = y + 0.30*torch.rand(y.size())

plt.figure(figsize=(8,4))
x_plot, y_plot = x.numpy(), y.numpy()
plt.scatter(x_plot, y_plot, color="red")
plt.xlabel("x")
plt.ylabel("y")
plt.show("Data for Regression Analysis")
plt.show()

Piccola differenza: usiamo un altro tipo di activation function, chiamata _Leaky ReLU_:

<img src="https://docs.pytorch.org/docs/2.11/_images/LeakyReLU.png" width="450"/>

$$\text{Leaky ReLU}(x)=\begin{cases}
    x & x\ge0 \\
    \epsilon\cdot x & x<0
\end{cases}$$

Con $\epsilon=0.01$ di default.

In [ ]:
n_neur = 100

Net = torch.nn.Sequential(
      torch.nn.Linear(1, n_neur),
      torch.nn.LeakyReLU(),
      torch.nn.Linear(n_neur, 1))

optimizer = torch.optim.Adam(Net.parameters(), lr = 0.01)
loss_function = torch.nn.MSELoss()

&#9888; **Questo potrebbe utilizzare un po' di memoria!**

In [ ]:
image_list = []
Niter = 1500 + 1

fig, ax = plt.subplots(figsize=(15,10))

for it in range(Niter):
    y_pred = Net(x)
    loss = loss_function(y_pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if it % 50 == 0:
        print(f"Doing Iterations: {it} and Loss: {loss}")
    if it % 5 == 0:
        plt.cla()
        ax.set_xlabel('x', fontsize=32)
        ax.set_ylabel('y Predictied', fontsize=32)
        ax.set_xlim(-6, 6)
        ax.set_ylim(-1.8, 1.8)
        ax.scatter(x_plot, y_plot, color = "red")
        ax.plot(x_plot, y_pred.data.numpy(), 'k-', lw=4)
        ax.text(0, 1.3, 'It = %d,' %it, fontdict={'size': 22, 'color':  'black'})
        ax.text(2, 1.3, 'Loss = %.6f' % loss.data.numpy(),
            fontdict={'size': 22, 'color':  'black'})
        fig.canvas.draw()      
        w, h = fig.canvas.get_width_height()
        image = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
        image = image.reshape((h, w, 4))
        image = image[:, :, :3].copy()
        image_list.append(image) 

In [ ]:
print("### Salvo .gif... ###")
imageio.mimsave('./Sin_Regression_2.gif', image_list, fps=5)